# 04 — Fine-tune ResNet-18 (ImageNet pretrained)

Same architecture as notebook 03 (ResNet-18, 500 classes) — only the **initialisation** differs:
ImageNet-1K weights, then replace `fc` and fine-tune on our iNat subset.

**Where to run:** Google Colab **T4** (Nate's PC has no NVIDIA CUDA).

**Workflow**
1. Colab bootstrap (Drive path already set)
2. Setup + data + `build_model(..., pretrained=True)`
3. **1-epoch smoke test**
4. Full fine-tune with checkpoints + curves
5. Copy best checkpoint to Drive (Colab disks are ephemeral)

Citations: He et al., ResNet, CVPR 2016; ImageNet pretrained weights via torchvision.


## 0. Colab bootstrap


In [ ]:
# Colab bootstrap — mounts Drive, sets INAT_DATA_ROOT, installs deps.
# Safe no-op on local Cursor.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"IN_COLAB = {IN_COLAB}")

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_SUBSET = Path("/content/drive/MyDrive/comp9517/subset")
    REPO_DIR = Path("/content/Haramcomp9517")
    BRANCH = "nate/pretrained-gradcam"
    REPO_URL = "https://github.com/NateRebello/Haramcomp9517.git"

    assert DRIVE_SUBSET.is_dir(), f"Missing {DRIVE_SUBSET} — check Drive path / shortcut"
    for split in ("train", "val", "test"):
        assert (DRIVE_SUBSET / split).is_dir(), f"Missing {DRIVE_SUBSET / split}"

    # Catch empty Drive stubs before ImageFolder's opaque error
    probe = DRIVE_SUBSET / "train" / "1022"
    if not probe.is_dir():
        # fall back to first class folder
        class_dirs = sorted(p for p in (DRIVE_SUBSET / "train").iterdir() if p.is_dir())
        assert class_dirs, f"No class folders under {DRIVE_SUBSET / 'train'}"
        probe = class_dirs[0]
    n_jpg = sum(1 for p in probe.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"probe {probe}: {n_jpg} images")
    assert n_jpg > 0, (
        f"Class folders exist but contain no images under {probe}. "
        "The Drive subset is incomplete (empty folders). Upload your full local "
        "Haramcomp9517/subset to MyDrive/comp9517/subset (must include *.jpg files)."
    )

    if not REPO_DIR.is_dir():
        # Fall back to main/deepl branch if Nate's branch is not on remote yet
        try:
            subprocess.check_call(
                ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)]
            )
        except subprocess.CalledProcessError:
            print(f"Branch {BRANCH} not on remote — cloning default branch")
            subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    else:
        subprocess.check_call(["git", "fetch", "origin"], cwd=REPO_DIR)

    pipeline = REPO_DIR / "dl_pipeline"
    # Prefer the uploaded notebook's sibling tree if present; else cloned repo
    os.chdir(pipeline)
    os.environ["INAT_DATA_ROOT"] = str(DRIVE_SUBSET)
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"]
    )
    if str(pipeline) not in sys.path:
        sys.path.insert(0, str(pipeline))
    print("cwd =", os.getcwd())
    print("INAT_DATA_ROOT =", os.environ["INAT_DATA_ROOT"])
else:
    # Local: default DATA_ROOT is repo-root/subset via src.config
    print("Local run — using default DATA_ROOT unless INAT_DATA_ROOT is set.")


## 1. Setup


In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.amp import GradScaler

PIPELINE_ROOT = Path.cwd().resolve()
if PIPELINE_ROOT.name == "notebooks":
    PIPELINE_ROOT = PIPELINE_ROOT.parent
if str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))

from src.config import (
    BATCH_SIZE,
    CHECKPOINTS_DIR,
    IMG_SIZE,
    NUM_CLASSES,
    RESULTS_DIR,
    SEED,
    ensure_output_dirs,
    set_seed,
)
from src.dataset import build_dataloaders, build_datasets
from src.models import build_model, count_parameters
from src.train_utils import (
    estimate_vram_mb,
    evaluate,
    save_checkpoint,
    save_history_json,
    train_one_epoch,
)

set_seed(SEED)
ensure_output_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GiB")
else:
    print("WARNING: no CUDA — fine-tuning on CPU is impractical. Use Colab T4.")


## 2. Run config

- **AdamW** + lower LR than from-scratch (`1e-4`): pretrained features need smaller steps.
- Same ImageNet normalisation / aug as notebook 03 so the comparison isolates transfer learning.
- Checkpoints go under `checkpoints/resnet18_pretrained/` (gitignored `*.pt`).


In [ ]:
# --- tweak here if needed ---
RUN_NAME = "resnet18_pretrained"
TRAIN_BATCH = BATCH_SIZE  # 16; drop to 8 on OOM
LR = 1e-4                 # lower than scratch (1e-3) for fine-tuning
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 15           # transfer usually converges faster than scratch
USE_AMP = True

CKPT_DIR = CHECKPOINTS_DIR / RUN_NAME
CKPT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CKPT = CKPT_DIR / "best.pt"
LAST_CKPT = CKPT_DIR / "last.pt"
HISTORY_JSON = RESULTS_DIR / f"{RUN_NAME}_history.json"

# Optional: also mirror best ckpt to Drive so Colab disconnects don't wipe it
DRIVE_CKPT_DIR = Path("/content/drive/MyDrive/comp9517/checkpoints/resnet18_pretrained")

run_config = {
    "run_name": RUN_NAME,
    "architecture": "resnet18",
    "pretrained": True,
    "num_classes": NUM_CLASSES,
    "img_size": IMG_SIZE,
    "batch_size": TRAIN_BATCH,
    "lr": LR,
    "weight_decay": WEIGHT_DECAY,
    "num_epochs": NUM_EPOCHS,
    "optimizer": "AdamW",
    "use_amp": USE_AMP,
    "seed": SEED,
}
print(run_config)


## 3. Data + model (`pretrained=True`)


In [ ]:
train_ds, val_ds, test_ds = build_datasets(augment_train=True)
train_loader, val_loader, test_loader = build_dataloaders(
    train_ds, val_ds, test_ds, batch_size=TRAIN_BATCH
)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print(f"batches/epoch (train)={len(train_loader)}")
print(f"DATA via config → see src.config / INAT_DATA_ROOT")

model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=True).to(device)
total_p, train_p = count_parameters(model)
print(f"params: total={total_p:,}  trainable={train_p:,}")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None


## 4. One-epoch smoke test

Run this **before** the full loop. After 1 epoch, pretrained top-1 should already beat random (~0.2%).


In [ ]:
print("=== SMOKE: 1 epoch (pretrained) ===")
t0 = time.perf_counter()
smoke_train = train_one_epoch(
    model, train_loader, criterion, optimizer, device, scaler, use_amp=USE_AMP
)
smoke_val = evaluate(model, val_loader, criterion, device, use_amp=USE_AMP)
alloc, reserved = estimate_vram_mb(device)
elapsed = time.perf_counter() - t0

print(f"train loss={smoke_train['loss']:.4f}  acc={smoke_train['acc']*100:.2f}%  ({smoke_train['seconds']:.1f}s)")
print(f"val   loss={smoke_val['loss']:.4f}  acc={smoke_val['acc']*100:.2f}%  ({smoke_val['seconds']:.1f}s)")
print(f"wall time 1 epoch ≈ {elapsed/60:.1f} min")
print(f"VRAM allocated={alloc:.0f} MiB  reserved={reserved:.0f} MiB")
print("Smoke OK — proceed to full training if loss is finite and no OOM.")


## 5. Full fine-tuning

Re-initialises with ImageNet weights so the smoke epoch does not pollute the logged run.
Saves `best.pt` (highest val top-1) and `last.pt` every epoch.


In [ ]:
# Fresh ImageNet init for the logged experiment
set_seed(SEED)
model = build_model("resnet18", num_classes=NUM_CLASSES, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler("cuda") if (USE_AMP and device.type == "cuda") else None

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_seconds": []}
best_val_acc = -1.0
train_start = time.perf_counter()

for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\n===== Epoch {epoch}/{NUM_EPOCHS} =====")
    tr = train_one_epoch(
        model, train_loader, criterion, optimizer, device, scaler, use_amp=USE_AMP
    )
    va = evaluate(model, val_loader, criterion, device, use_amp=USE_AMP)

    history["train_loss"].append(tr["loss"])
    history["train_acc"].append(tr["acc"])
    history["val_loss"].append(va["loss"])
    history["val_acc"].append(va["acc"])
    history["epoch_seconds"].append(tr["seconds"] + va["seconds"])

    print(
        f"train loss={tr['loss']:.4f} acc={tr['acc']*100:.2f}% | "
        f"val loss={va['loss']:.4f} acc={va['acc']*100:.2f}% | "
        f"{history['epoch_seconds'][-1]:.0f}s"
    )

    save_checkpoint(
        LAST_CKPT,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
        history=history,
        config=run_config,
        class_to_idx=train_ds.class_to_idx,
        best_val_acc=best_val_acc,
    )

    if va["acc"] > best_val_acc:
        best_val_acc = va["acc"]
        save_checkpoint(
            BEST_CKPT,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            history=history,
            config=run_config,
            class_to_idx=train_ds.class_to_idx,
            best_val_acc=best_val_acc,
        )
        print(f"  new best val acc={best_val_acc*100:.2f}% → {BEST_CKPT.name}")

total_train_s = time.perf_counter() - train_start
run_config["total_train_seconds"] = total_train_s
run_config["best_val_acc"] = best_val_acc
save_history_json(HISTORY_JSON, history, run_config)

# Mirror to Drive (Colab)
if IN_COLAB:
    import shutil

    DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for src in (BEST_CKPT, LAST_CKPT, HISTORY_JSON):
        if src.is_file():
            dst = DRIVE_CKPT_DIR / src.name if src.suffix == ".pt" else Path("/content/drive/MyDrive/comp9517/results") / src.name
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f"copied → {dst}")

print(f"\nDone. best val acc={best_val_acc*100:.2f}%  total time={total_train_s/60:.1f} min")
print(f"history → {HISTORY_JSON}")
print(f"best ckpt → {BEST_CKPT}")


## 6. Training curves


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(epochs, history["train_loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="val")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("loss")
axes[0].set_title("Loss (ImageNet pretrained fine-tune)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="train")
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]], label="val")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("top-1 acc (%)")
axes[1].set_title("Accuracy (ImageNet pretrained fine-tune)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
curve_path = RESULTS_DIR / f"{RUN_NAME}_curves.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print(f"saved {curve_path}")

if IN_COLAB:
    import shutil

    drive_results = Path("/content/drive/MyDrive/comp9517/results")
    drive_results.mkdir(parents=True, exist_ok=True)
    shutil.copy2(curve_path, drive_results / curve_path.name)
    print(f"copied curves → {drive_results / curve_path.name}")


## 7. Next

1. After a full run, evaluate with `05_evaluation.ipynb` (pretrained checkpoint path is wired in).
2. Then Grad-CAM: `06_gradcam_analysis.ipynb`.

Keep **Runtime → Prevent sleeping** / avoid closing the tab during the full train.
